# Audit Compliance Agent
### Automated Policy Evidence Collection, Log Aggregation & Audit-Package Assembly

**Project — NLP / Agentic Workflows Portfolio**

This notebook builds an **agent-based audit compliance system** that:
1. Gathers **policy evidence** from document stores
2. Collects and structures **system logs** with timestamps
3. Attaches **reviewer notes and annotations**
4. Validates completeness against a **compliance checklist**
5. Assembles everything into a **traceable, audit-ready package**

Every action taken by every agent is recorded in a tamper-evident **audit trail** with cryptographic hashes, so auditors can verify the full chain of custody.

## Project Overview

Regulatory audits (SOC 2, ISO 27001, HIPAA, PCI-DSS) require organisations to prove that policies are in place, systems behave as documented, and reviews have been conducted. Manually collecting this evidence is tedious and error-prone.

This notebook demonstrates an **agentic workflow** where specialised agents collaborate to:

| Agent | Responsibility |
|---|---|
| **PolicyAgent** | Searches policy documents, retrieves relevant sections, verifies version currency |
| **LogAgent** | Queries system/application logs, filters by time window and severity, structures entries |
| **NoteAgent** | Collects reviewer annotations, links notes to specific controls |
| **ChecklistAgent** | Validates that all required controls have sufficient evidence |
| **PackageAgent** | Assembles the final audit package with table of contents, evidence index, and hash chain |

### Why Traceability Matters

Auditors do not just want evidence — they want **proof that the evidence was collected correctly**. Every agent action is logged with:
- Timestamp (ISO 8601)
- Agent identity
- Action description
- Input hash (what the agent received)
- Output hash (what the agent produced)
- Status and any warnings

## Learning Objectives

1. Design a **multi-agent workflow** with clear separation of concerns
2. Implement a **cryptographic audit trail** using SHA-256 hash chains
3. Build a **compliance checklist engine** that maps controls to evidence
4. Demonstrate **structured log parsing** with severity filtering and time windowing
5. Assemble a complete **audit package** with cross-referenced evidence index
6. Understand how traceability requirements shape agent architecture

## Problem Statement

Given a set of **compliance controls** (e.g. "Access reviews are conducted quarterly"), automatically:
1. Locate the **governing policy** document and extract the relevant section
2. Retrieve **system logs** that demonstrate the control was executed
3. Collect any **reviewer notes** or sign-off annotations
4. **Validate** that each control has sufficient evidence (policy + log + note)
5. Package everything into a **single audit-ready artifact** with full traceability

## Why This Matters

- Manual audit preparation takes **40-200 hours per audit cycle** for mid-size organisations
- Missing evidence for a single control can trigger a **qualified opinion** or finding
- Hash-chained audit trails provide **non-repudiation** — evidence cannot be silently altered after collection
- Agentic automation reduces human error in evidence gathering while maintaining the reviewer-in-the-loop for judgement calls

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ["pandas", "numpy", "matplotlib", "seaborn", "plotly"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Environment ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import hashlib, json, uuid, textwrap, re, copy
from datetime import datetime, timedelta, timezone
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import Any
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK")

## Configuration & Constants

In [ ]:
# Audit configuration
AUDIT_ID        = "AUD-2026-Q1-0042"
AUDIT_FRAMEWORK = "SOC 2 Type II"
AUDIT_PERIOD    = ("2025-10-01", "2025-12-31")   # Q4 2025
ORGANISATION    = "Acme Cloud Services Inc."
AUDITOR         = "External Audit Partners LLP"
PREPARER        = "audit-compliance-agent/v1.0"

# Time window for log queries
LOG_START = datetime(2025, 10, 1, tzinfo=timezone.utc)
LOG_END   = datetime(2025, 12, 31, 23, 59, 59, tzinfo=timezone.utc)

# Evidence sufficiency thresholds
MIN_POLICIES_PER_CONTROL = 1
MIN_LOG_ENTRIES_PER_CONTROL = 1
MIN_NOTES_PER_CONTROL = 0  # notes are optional but encouraged

print(f"Audit: {AUDIT_ID}")
print(f"Framework: {AUDIT_FRAMEWORK}")
print(f"Period: {AUDIT_PERIOD[0]} to {AUDIT_PERIOD[1]}")
print(f"Organisation: {ORGANISATION}")

## Audit Trail Infrastructure

The audit trail is the backbone of traceability. Every agent action is recorded as an `AuditEntry` that includes:
- A **sequential ID** for ordering
- **Timestamps** in UTC ISO 8601
- The **agent** that performed the action
- A **hash of the input** and **hash of the output** (SHA-256)
- A **chain hash** linking this entry to the previous one (like a blockchain)

This makes the trail **tamper-evident**: changing any entry invalidates all subsequent chain hashes.

In [ ]:
@dataclass
class AuditEntry:
    """Single entry in the audit trail."""
    seq: int
    timestamp: str
    agent: str
    action: str
    input_hash: str
    output_hash: str
    chain_hash: str
    status: str = "OK"
    details: str = ""


class AuditTrail:
    """Append-only, hash-chained audit trail."""

    def __init__(self, audit_id: str):
        self.audit_id = audit_id
        self.entries: list[AuditEntry] = []
        self._seq = 0
        # Genesis hash
        self._prev_hash = hashlib.sha256(
            f"GENESIS|{audit_id}".encode()
        ).hexdigest()

    @staticmethod
    def _hash(data: str) -> str:
        return hashlib.sha256(data.encode("utf-8")).hexdigest()

    def log(self, agent: str, action: str, input_data: str,
            output_data: str, status: str = "OK", details: str = "") -> AuditEntry:
        self._seq += 1
        ts = datetime.now(timezone.utc).isoformat()
        in_hash  = self._hash(input_data)
        out_hash = self._hash(output_data)

        # Chain hash = H(prev_hash | seq | agent | action | in_hash | out_hash)
        chain_input = f"{self._prev_hash}|{self._seq}|{agent}|{action}|{in_hash}|{out_hash}"
        chain_hash  = self._hash(chain_input)

        entry = AuditEntry(
            seq=self._seq, timestamp=ts, agent=agent, action=action,
            input_hash=in_hash, output_hash=out_hash,
            chain_hash=chain_hash, status=status, details=details,
        )
        self.entries.append(entry)
        self._prev_hash = chain_hash
        return entry

    def verify_chain(self) -> tuple[bool, list[str]]:
        """Verify the entire chain. Returns (valid, errors)."""
        errors = []
        prev = self._hash(f"GENESIS|{self.audit_id}")
        for e in self.entries:
            expected = self._hash(
                f"{prev}|{e.seq}|{e.agent}|{e.action}|{e.input_hash}|{e.output_hash}"
            )
            if expected != e.chain_hash:
                errors.append(f"Seq {e.seq}: chain hash mismatch")
            prev = e.chain_hash
        return (len(errors) == 0, errors)

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([asdict(e) for e in self.entries])

    def __len__(self):
        return len(self.entries)


# Initialise the global audit trail
trail = AuditTrail(AUDIT_ID)
print(f"Audit trail initialised for {AUDIT_ID}")
print(f"Genesis hash: {trail._prev_hash[:24]}...")

## Compliance Controls Definition

We define the **controls** that must be evidenced during this audit. Each control maps to one or more Trust Service Criteria (TSC) in SOC 2.

In [ ]:
class ControlStatus(Enum):
    NOT_STARTED  = "not_started"
    IN_PROGRESS  = "in_progress"
    EVIDENCED    = "evidenced"
    INSUFFICIENT = "insufficient"
    EXCEPTION    = "exception"


@dataclass
class ComplianceControl:
    control_id: str
    title: str
    description: str
    tsc_criteria: list[str]        # SOC 2 Trust Service Criteria
    evidence_types: list[str]      # what evidence is needed
    status: ControlStatus = ControlStatus.NOT_STARTED
    policies: list[dict] = field(default_factory=list)
    logs: list[dict] = field(default_factory=list)
    notes: list[dict] = field(default_factory=list)


# Define 8 controls covering key SOC 2 areas
controls = {
    "CC-6.1": ComplianceControl(
        control_id="CC-6.1",
        title="Logical Access Controls",
        description="Access to systems is restricted through logical access security measures.",
        tsc_criteria=["CC6.1", "CC6.2"],
        evidence_types=["policy", "log", "note"],
    ),
    "CC-6.3": ComplianceControl(
        control_id="CC-6.3",
        title="Access Removal on Termination",
        description="Access is revoked for terminated employees within 24 hours.",
        tsc_criteria=["CC6.3"],
        evidence_types=["policy", "log", "note"],
    ),
    "CC-7.1": ComplianceControl(
        control_id="CC-7.1",
        title="Vulnerability Management",
        description="Systems are scanned for vulnerabilities monthly and critical findings remediated within 30 days.",
        tsc_criteria=["CC7.1"],
        evidence_types=["policy", "log"],
    ),
    "CC-7.2": ComplianceControl(
        control_id="CC-7.2",
        title="Security Incident Response",
        description="Security incidents are detected, reported, and resolved per the incident response plan.",
        tsc_criteria=["CC7.2", "CC7.3"],
        evidence_types=["policy", "log", "note"],
    ),
    "CC-8.1": ComplianceControl(
        control_id="CC-8.1",
        title="Change Management",
        description="Changes to production are authorised, tested, and approved before deployment.",
        tsc_criteria=["CC8.1"],
        evidence_types=["policy", "log"],
    ),
    "A-1.2": ComplianceControl(
        control_id="A-1.2",
        title="Backup and Recovery",
        description="System backups are performed daily and tested quarterly.",
        tsc_criteria=["A1.2"],
        evidence_types=["policy", "log", "note"],
    ),
    "CC-3.1": ComplianceControl(
        control_id="CC-3.1",
        title="Risk Assessment",
        description="Annual risk assessment identifies and prioritises threats to operations.",
        tsc_criteria=["CC3.1", "CC3.2"],
        evidence_types=["policy", "note"],
    ),
    "CC-1.4": ComplianceControl(
        control_id="CC-1.4",
        title="Security Awareness Training",
        description="All employees complete security awareness training annually.",
        tsc_criteria=["CC1.4"],
        evidence_types=["policy", "log"],
    ),
}

print(f"Defined {len(controls)} compliance controls:")
for cid, ctrl in controls.items():
    print(f"  {cid}: {ctrl.title} (evidence: {', '.join(ctrl.evidence_types)})")

## Simulated Data Sources

In production, these would be API calls to document management systems, SIEM tools, and ticketing platforms. Here we simulate realistic data that mirrors enterprise audit evidence.

In [ ]:
# ── Policy Document Store ──────────────────────────────────────────
POLICY_STORE = {
    "POL-IAM-001": {
        "doc_id": "POL-IAM-001",
        "title": "Identity and Access Management Policy",
        "version": "3.2",
        "effective_date": "2025-01-15",
        "owner": "CISO Office",
        "status": "active",
        "sections": {
            "4.1": "All access to production systems shall require multi-factor authentication (MFA). "
                   "Service accounts are exempt only when compensating controls (IP allowlisting, "
                   "short-lived tokens) are documented and approved by the Security team.",
            "4.2": "Access reviews shall be conducted quarterly by system owners. Results are "
                   "documented in the GRC tool and deviations reported to the Risk Committee.",
            "4.3": "Upon employee termination, HR triggers an automated workflow that disables "
                   "all accounts within 24 hours. The Identity team verifies completion within 48 hours.",
            "5.1": "Privileged access (admin, root, DBA) requires a separate approval from the "
                   "Security Director and is limited to 90-day renewable grants.",
        },
        "controls_mapped": ["CC-6.1", "CC-6.3"],
    },
    "POL-VM-002": {
        "doc_id": "POL-VM-002",
        "title": "Vulnerability Management Policy",
        "version": "2.1",
        "effective_date": "2025-03-01",
        "owner": "Security Operations",
        "status": "active",
        "sections": {
            "3.1": "All internet-facing and internal production systems shall be scanned for "
                   "vulnerabilities at least monthly using approved scanning tools.",
            "3.2": "Critical vulnerabilities (CVSS >= 9.0) must be remediated within 14 days. "
                   "High vulnerabilities (CVSS 7.0-8.9) within 30 days.",
            "3.3": "Scan results and remediation status are reported to the Security Committee monthly.",
        },
        "controls_mapped": ["CC-7.1"],
    },
    "POL-IR-003": {
        "doc_id": "POL-IR-003",
        "title": "Incident Response Plan",
        "version": "4.0",
        "effective_date": "2025-02-10",
        "owner": "Security Operations",
        "status": "active",
        "sections": {
            "2.1": "Security incidents are classified into Severity 1 (critical), Severity 2 (high), "
                   "Severity 3 (medium), and Severity 4 (low) based on impact and scope.",
            "2.2": "Severity 1 incidents require executive notification within 1 hour and "
                   "post-incident review within 5 business days.",
            "3.1": "The incident response team conducts quarterly tabletop exercises to validate "
                   "the response plan. Results are documented and gaps addressed.",
        },
        "controls_mapped": ["CC-7.2"],
    },
    "POL-CM-004": {
        "doc_id": "POL-CM-004",
        "title": "Change Management Policy",
        "version": "2.5",
        "effective_date": "2025-04-01",
        "owner": "Engineering",
        "status": "active",
        "sections": {
            "3.1": "All production changes require a Change Request (CR) approved by the change "
                   "advisory board (CAB) or designated approver before deployment.",
            "3.2": "Emergency changes bypass CAB but require retrospective approval within 48 hours "
                   "and a root cause document.",
            "4.1": "Changes are tested in a staging environment. Test evidence (screenshots, logs) "
                   "is attached to the CR before production deployment.",
        },
        "controls_mapped": ["CC-8.1"],
    },
    "POL-BCP-005": {
        "doc_id": "POL-BCP-005",
        "title": "Business Continuity and Backup Policy",
        "version": "1.8",
        "effective_date": "2025-05-15",
        "owner": "Infrastructure",
        "status": "active",
        "sections": {
            "2.1": "Daily incremental backups and weekly full backups are performed for all "
                   "production databases and file stores.",
            "2.2": "Backup restoration is tested quarterly. Tests include full restoration to "
                   "an isolated environment and data integrity verification.",
            "3.1": "Backup retention: 30 days for incremental, 90 days for full, 7 years for "
                   "annual compliance snapshots.",
        },
        "controls_mapped": ["A-1.2"],
    },
    "POL-RA-006": {
        "doc_id": "POL-RA-006",
        "title": "Enterprise Risk Assessment Framework",
        "version": "2.0",
        "effective_date": "2025-01-01",
        "owner": "Risk Management",
        "status": "active",
        "sections": {
            "2.1": "An enterprise-wide risk assessment is conducted annually. The assessment "
                   "identifies threats, vulnerabilities, and likelihood/impact ratings.",
            "2.2": "Risk treatment plans are developed for all risks rated High or Critical. "
                   "Plans include responsible owner, target date, and residual risk acceptance.",
            "3.1": "The Risk Register is reviewed quarterly by the Risk Committee and updated "
                   "with any new threats or changes in the control environment.",
        },
        "controls_mapped": ["CC-3.1"],
    },
    "POL-SAT-007": {
        "doc_id": "POL-SAT-007",
        "title": "Security Awareness Training Policy",
        "version": "1.5",
        "effective_date": "2025-06-01",
        "owner": "People Operations",
        "status": "active",
        "sections": {
            "2.1": "All employees and contractors must complete annual security awareness training "
                   "within 30 days of hire and annually thereafter.",
            "2.2": "Training covers phishing recognition, password hygiene, data classification, "
                   "and incident reporting procedures.",
            "3.1": "Completion rates are tracked in the LMS. Departments below 95% completion "
                   "are escalated to the CISO for remediation.",
        },
        "controls_mapped": ["CC-1.4"],
    },
}

print(f"Policy store: {len(POLICY_STORE)} documents")
for doc_id, doc in POLICY_STORE.items():
    print(f"  {doc_id}: {doc['title']} (v{doc['version']}, maps to {doc['controls_mapped']})")

In [ ]:
# ── System Log Store ───────────────────────────────────────────────
# Simulated structured logs from various systems
np.random.seed(42)

def generate_logs(n_logs=200):
    """Generate realistic system/application logs for the audit period."""
    log_templates = [
        {"source": "IAM-System", "category": "access_control",
         "controls": ["CC-6.1"],
         "templates": [
             ("User {user} authenticated via MFA from {ip}", "INFO"),
             ("Quarterly access review completed for {system}: {n} accounts reviewed", "INFO"),
             ("Privileged access granted to {user} for {system} — approved by {approver}", "INFO"),
             ("Failed login attempt for {user} from {ip} — account locked after 5 attempts", "WARN"),
         ]},
        {"source": "IAM-System", "category": "access_removal",
         "controls": ["CC-6.3"],
         "templates": [
             ("Employee {user} terminated — all accounts disabled within {hours}h", "INFO"),
             ("Termination workflow completed for {user}: {n} systems deprovisioned", "INFO"),
             ("ALERT: Account for terminated employee {user} still active after 24h", "WARN"),
         ]},
        {"source": "Vuln-Scanner", "category": "vulnerability",
         "controls": ["CC-7.1"],
         "templates": [
             ("Monthly scan completed: {n_crit} critical, {n_high} high, {n_med} medium findings", "INFO"),
             ("Critical CVE-{cve} remediated on {system} within {days} days", "INFO"),
             ("Overdue: CVE-{cve} on {system} — {days} days past SLA", "WARN"),
         ]},
        {"source": "SIEM", "category": "incident",
         "controls": ["CC-7.2"],
         "templates": [
             ("Incident INC-{ticket} opened: {desc} — Severity {sev}", "INFO"),
             ("Incident INC-{ticket} resolved after {hours}h — root cause: {cause}", "INFO"),
             ("Tabletop exercise completed: {n} participants, {gaps} gaps identified", "INFO"),
         ]},
        {"source": "CI-CD", "category": "change_management",
         "controls": ["CC-8.1"],
         "templates": [
             ("CR-{ticket} approved by {approver} — deploying {service} v{ver}", "INFO"),
             ("CR-{ticket} deployed to production — all tests passed", "INFO"),
             ("Emergency change EC-{ticket} deployed — retrospective approval pending", "WARN"),
         ]},
        {"source": "Backup-System", "category": "backup",
         "controls": ["A-1.2"],
         "templates": [
             ("Daily incremental backup completed: {db} — {size}GB, {duration}min", "INFO"),
             ("Weekly full backup completed: {db} — {size}GB", "INFO"),
             ("Quarterly restore test: {db} restored successfully — integrity verified", "INFO"),
             ("Backup FAILED for {db}: {reason}", "ERROR"),
         ]},
        {"source": "LMS", "category": "training",
         "controls": ["CC-1.4"],
         "templates": [
             ("Security awareness training completed by {user} — score: {score}%", "INFO"),
             ("Training completion rate for {dept}: {rate}%", "INFO"),
             ("REMINDER: {n} employees overdue for security training", "WARN"),
         ]},
    ]

    users = ["jsmith", "agarcia", "mchen", "kpatel", "lwilson",
             "rjohnson", "dnguyen", "bthomas", "swright", "jlee"]
    systems = ["prod-api", "prod-db", "staging-k8s", "internal-wiki", "crm-app"]
    dbs = ["orders-db", "users-db", "analytics-dw", "config-store"]
    services = ["auth-service", "payment-api", "notification-svc", "dashboard"]
    depts = ["Engineering", "Sales", "Marketing", "Finance", "Support"]

    logs = []
    for _ in range(n_logs):
        group = np.random.choice(log_templates)
        tmpl, severity = group["templates"][np.random.randint(len(group["templates"]))]
        ts = LOG_START + timedelta(
            seconds=np.random.randint(0, int((LOG_END - LOG_START).total_seconds()))
        )
        msg = tmpl.format(
            user=np.random.choice(users), ip=f"10.0.{np.random.randint(1,255)}.{np.random.randint(1,255)}",
            system=np.random.choice(systems), n=np.random.randint(5, 150),
            approver=np.random.choice(users), hours=np.random.choice([2, 4, 8, 12, 18, 26]),
            n_crit=np.random.randint(0, 5), n_high=np.random.randint(0, 12),
            n_med=np.random.randint(5, 30), cve=f"2025-{np.random.randint(10000,99999)}",
            days=np.random.randint(1, 45), ticket=np.random.randint(1000, 9999),
            desc=np.random.choice(["Suspicious login activity", "DDoS attempt detected",
                                    "Data exfiltration alert", "Malware detection"]),
            sev=np.random.choice([1, 2, 3, 4]), cause=np.random.choice(
                ["misconfigured firewall", "phishing", "software bug", "third-party breach"]),
            gaps=np.random.randint(0, 4), service=np.random.choice(services),
            ver=f"{np.random.randint(1,5)}.{np.random.randint(0,20)}.{np.random.randint(0,99)}",
            db=np.random.choice(dbs), size=np.random.randint(10, 500),
            duration=np.random.randint(5, 120),
            reason=np.random.choice(["disk full", "network timeout", "permission denied"]),
            score=np.random.randint(70, 100), dept=np.random.choice(depts),
            rate=np.random.randint(80, 100),
        )
        logs.append({
            "timestamp": ts.isoformat(),
            "source": group["source"],
            "category": group["category"],
            "severity": severity,
            "message": msg,
            "controls": group["controls"],
        })

    return sorted(logs, key=lambda x: x["timestamp"])


LOG_STORE = generate_logs(200)
print(f"Generated {len(LOG_STORE)} log entries")
sev_counts = defaultdict(int)
for l in LOG_STORE:
    sev_counts[l["severity"]] += 1
print(f"Severity distribution: {dict(sev_counts)}")

In [ ]:
# ── Reviewer Notes Store ───────────────────────────────────────────
NOTES_STORE = [
    {
        "note_id": "N-001",
        "author": "Maria Chen — IT Compliance Manager",
        "date": "2025-11-15",
        "control_id": "CC-6.1",
        "note": "Q4 access review completed on time. Two stale service accounts identified "
                "and disabled. No policy exceptions required.",
        "disposition": "satisfactory",
    },
    {
        "note_id": "N-002",
        "author": "James Wright — Security Operations Lead",
        "date": "2025-12-01",
        "control_id": "CC-6.3",
        "note": "Reviewed termination logs for Q4. One instance (employee rjohnson) exceeded "
                "the 24-hour SLA by 2 hours due to a weekend termination. Root cause documented "
                "and weekend on-call process updated.",
        "disposition": "minor_finding",
    },
    {
        "note_id": "N-003",
        "author": "Sarah Patel — Vulnerability Management",
        "date": "2025-12-10",
        "control_id": "CC-7.1",
        "note": "All three monthly scans completed on schedule. October scan had 2 critical "
                "findings; both remediated within 10 days (SLA: 14 days).",
        "disposition": "satisfactory",
    },
    {
        "note_id": "N-004",
        "author": "David Kim — Incident Commander",
        "date": "2025-12-20",
        "control_id": "CC-7.2",
        "note": "Q4 tabletop exercise conducted in November. Scenario: ransomware on CI/CD. "
                "Two communication gaps identified and remediated before year-end.",
        "disposition": "satisfactory",
    },
    {
        "note_id": "N-005",
        "author": "Lisa Johnson — Change Advisory Board Chair",
        "date": "2025-12-15",
        "control_id": "CC-8.1",
        "note": "Reviewed all emergency changes in Q4. Three emergency CRs filed; all received "
                "retrospective approval within 48 hours per policy. No unauthorized changes detected.",
        "disposition": "satisfactory",
    },
    {
        "note_id": "N-006",
        "author": "Robert Nguyen — Infrastructure Lead",
        "date": "2025-12-18",
        "control_id": "A-1.2",
        "note": "Quarterly backup restore test completed successfully on Dec 5. Full restoration "
                "of orders-db took 47 minutes. Integrity hash verified against live checksum.",
        "disposition": "satisfactory",
    },
    {
        "note_id": "N-007",
        "author": "Elena Vasquez — Chief Risk Officer",
        "date": "2025-11-20",
        "control_id": "CC-3.1",
        "note": "Annual risk assessment completed in November 2025. Three new risks added to the "
                "register (AI model drift, third-party API dependency, supply chain attacks). "
                "All rated High with treatment plans assigned.",
        "disposition": "satisfactory",
    },
    {
        "note_id": "N-008",
        "author": "Tom Garcia — People Operations",
        "date": "2025-12-30",
        "control_id": "CC-1.4",
        "note": "Annual security awareness training completion rate: 97.2% as of Dec 30. "
                "Remaining 2.8% (4 employees) are on extended leave; completion deadline "
                "extended to Jan 31 per policy exception.",
        "disposition": "minor_finding",
    },
]

print(f"Reviewer notes: {len(NOTES_STORE)} entries")
for n in NOTES_STORE:
    print(f"  {n['note_id']}: {n['control_id']} — {n['disposition']}")

## Agent Definitions

Each agent is a Python class with a well-defined interface:
- `collect(control: ComplianceControl) -> list[dict]` — gather evidence for one control
- All agents receive a reference to the global `AuditTrail` and log every action

### PolicyAgent

Searches the policy document store, retrieves sections relevant to a given control, and verifies document currency (version, effective date).

In [ ]:
class PolicyAgent:
    """Searches policy documents and retrieves relevant sections for a control."""

    NAME = "PolicyAgent"

    def __init__(self, policy_store: dict, audit_trail: AuditTrail):
        self.store = policy_store
        self.trail = audit_trail

    def collect(self, control: ComplianceControl) -> list[dict]:
        """Find all policy sections mapped to the given control."""
        results = []
        for doc_id, doc in self.store.items():
            if control.control_id in doc.get("controls_mapped", []):
                for sec_id, sec_text in doc["sections"].items():
                    evidence = {
                        "evidence_type": "policy",
                        "doc_id": doc_id,
                        "doc_title": doc["title"],
                        "version": doc["version"],
                        "effective_date": doc["effective_date"],
                        "section": sec_id,
                        "text": sec_text,
                        "control_id": control.control_id,
                        "collected_at": datetime.now(timezone.utc).isoformat(),
                    }
                    results.append(evidence)

                # Log the collection action
                self.trail.log(
                    agent=self.NAME,
                    action=f"collect_policy:{doc_id}",
                    input_data=f"control={control.control_id}",
                    output_data=json.dumps({"doc_id": doc_id, "sections": len(doc["sections"])}),
                    details=f"Retrieved {len(doc['sections'])} sections from {doc['title']}",
                )

        if not results:
            self.trail.log(
                agent=self.NAME,
                action=f"collect_policy:NOT_FOUND",
                input_data=f"control={control.control_id}",
                output_data="[]",
                status="WARN",
                details=f"No policy documents mapped to {control.control_id}",
            )

        return results


policy_agent = PolicyAgent(POLICY_STORE, trail)
print(f"PolicyAgent initialised with {len(POLICY_STORE)} documents")

### LogAgent

Queries the system log store, filters entries by control mapping, time window, and severity, then structures them as evidence.

In [ ]:
class LogAgent:
    """Queries and filters system logs for audit evidence."""

    NAME = "LogAgent"

    def __init__(self, log_store: list[dict], audit_trail: AuditTrail):
        self.store = log_store
        self.trail = audit_trail

    def collect(self, control: ComplianceControl,
                min_severity: str = "INFO") -> list[dict]:
        """Retrieve logs relevant to the given control."""
        severity_order = {"INFO": 0, "WARN": 1, "ERROR": 2, "CRITICAL": 3}
        min_sev = severity_order.get(min_severity, 0)

        results = []
        for log in self.store:
            if control.control_id not in log.get("controls", []):
                continue
            if severity_order.get(log["severity"], 0) < min_sev:
                continue

            evidence = {
                "evidence_type": "log",
                "timestamp": log["timestamp"],
                "source": log["source"],
                "category": log["category"],
                "severity": log["severity"],
                "message": log["message"],
                "control_id": control.control_id,
                "collected_at": datetime.now(timezone.utc).isoformat(),
            }
            results.append(evidence)

        # Log the query
        self.trail.log(
            agent=self.NAME,
            action=f"collect_logs:{control.control_id}",
            input_data=f"control={control.control_id},min_severity={min_severity}",
            output_data=json.dumps({"count": len(results)}),
            details=f"Retrieved {len(results)} log entries for {control.control_id}",
        )

        return results


log_agent = LogAgent(LOG_STORE, trail)
print(f"LogAgent initialised with {len(LOG_STORE)} log entries")

### NoteAgent

Retrieves reviewer notes and annotations linked to specific controls. These provide the human judgement layer that auditors value.

In [ ]:
class NoteAgent:
    """Collects reviewer notes and annotations for a control."""

    NAME = "NoteAgent"

    def __init__(self, notes_store: list[dict], audit_trail: AuditTrail):
        self.store = notes_store
        self.trail = audit_trail

    def collect(self, control: ComplianceControl) -> list[dict]:
        """Retrieve notes linked to the given control."""
        results = []
        for note in self.store:
            if note["control_id"] == control.control_id:
                evidence = {
                    "evidence_type": "note",
                    "note_id": note["note_id"],
                    "author": note["author"],
                    "date": note["date"],
                    "text": note["note"],
                    "disposition": note["disposition"],
                    "control_id": control.control_id,
                    "collected_at": datetime.now(timezone.utc).isoformat(),
                }
                results.append(evidence)

        self.trail.log(
            agent=self.NAME,
            action=f"collect_notes:{control.control_id}",
            input_data=f"control={control.control_id}",
            output_data=json.dumps({"count": len(results)}),
            details=f"Retrieved {len(results)} notes for {control.control_id}",
        )

        return results


note_agent = NoteAgent(NOTES_STORE, trail)
print(f"NoteAgent initialised with {len(NOTES_STORE)} notes")

### ChecklistAgent

Validates that each control has sufficient evidence across all required categories. Flags insufficient controls.

In [ ]:
class ChecklistAgent:
    """Validates evidence sufficiency for each control."""

    NAME = "ChecklistAgent"

    def __init__(self, audit_trail: AuditTrail):
        self.trail = audit_trail

    def validate(self, control: ComplianceControl) -> dict:
        """Check whether the control has met all evidence requirements."""
        checks = {}
        all_pass = True

        if "policy" in control.evidence_types:
            has_policy = len(control.policies) >= MIN_POLICIES_PER_CONTROL
            checks["policy"] = {
                "required": True,
                "found": len(control.policies),
                "minimum": MIN_POLICIES_PER_CONTROL,
                "pass": has_policy,
            }
            if not has_policy:
                all_pass = False

        if "log" in control.evidence_types:
            has_logs = len(control.logs) >= MIN_LOG_ENTRIES_PER_CONTROL
            checks["log"] = {
                "required": True,
                "found": len(control.logs),
                "minimum": MIN_LOG_ENTRIES_PER_CONTROL,
                "pass": has_logs,
            }
            if not has_logs:
                all_pass = False

        if "note" in control.evidence_types:
            has_notes = len(control.notes) >= MIN_NOTES_PER_CONTROL
            checks["note"] = {
                "required": True,
                "found": len(control.notes),
                "minimum": MIN_NOTES_PER_CONTROL,
                "pass": has_notes,
            }
            if not has_notes:
                all_pass = False

        # Determine control status
        has_minor = any(
            n.get("disposition") == "minor_finding" for n in control.notes
        )
        if all_pass and not has_minor:
            control.status = ControlStatus.EVIDENCED
        elif all_pass and has_minor:
            control.status = ControlStatus.EVIDENCED  # evidenced with findings
        else:
            control.status = ControlStatus.INSUFFICIENT

        result = {
            "control_id": control.control_id,
            "status": control.status.value,
            "checks": checks,
            "all_pass": all_pass,
            "has_minor_findings": has_minor,
        }

        self.trail.log(
            agent=self.NAME,
            action=f"validate:{control.control_id}",
            input_data=json.dumps({
                "policies": len(control.policies),
                "logs": len(control.logs),
                "notes": len(control.notes),
            }),
            output_data=json.dumps(result),
            status="OK" if all_pass else "WARN",
            details=f"{'PASS' if all_pass else 'INSUFFICIENT'} — {control.title}",
        )

        return result


checklist_agent = ChecklistAgent(trail)
print("ChecklistAgent initialised")

### PackageAgent

Assembles the final audit package — a structured dictionary with:
- Cover page metadata
- Evidence index (cross-referenced by control)
- Full audit trail with chain verification
- Summary statistics and findings

In [ ]:
class PackageAgent:
    """Assembles the final audit-ready package."""

    NAME = "PackageAgent"

    def __init__(self, audit_trail: AuditTrail):
        self.trail = audit_trail

    def assemble(self, controls: dict[str, ComplianceControl],
                 validation_results: list[dict]) -> dict:
        """Build the complete audit package."""

        # Cover page
        cover = {
            "audit_id": AUDIT_ID,
            "framework": AUDIT_FRAMEWORK,
            "organisation": ORGANISATION,
            "audit_period": AUDIT_PERIOD,
            "auditor": AUDITOR,
            "prepared_by": PREPARER,
            "prepared_at": datetime.now(timezone.utc).isoformat(),
            "total_controls": len(controls),
        }

        # Evidence index
        evidence_index = []
        for cid, ctrl in controls.items():
            entry = {
                "control_id": cid,
                "title": ctrl.title,
                "status": ctrl.status.value,
                "tsc_criteria": ctrl.tsc_criteria,
                "policies_count": len(ctrl.policies),
                "logs_count": len(ctrl.logs),
                "notes_count": len(ctrl.notes),
                "total_evidence": len(ctrl.policies) + len(ctrl.logs) + len(ctrl.notes),
            }
            evidence_index.append(entry)

        # Summary statistics
        statuses = [ctrl.status.value for ctrl in controls.values()]
        summary = {
            "evidenced": statuses.count("evidenced"),
            "insufficient": statuses.count("insufficient"),
            "not_started": statuses.count("not_started"),
            "total_policies": sum(len(c.policies) for c in controls.values()),
            "total_logs": sum(len(c.logs) for c in controls.values()),
            "total_notes": sum(len(c.notes) for c in controls.values()),
            "minor_findings": sum(
                1 for c in controls.values()
                for n in c.notes if n.get("disposition") == "minor_finding"
            ),
        }

        # Audit trail verification
        chain_valid, chain_errors = self.trail.verify_chain()

        package = {
            "cover": cover,
            "evidence_index": evidence_index,
            "validation_results": validation_results,
            "summary": summary,
            "audit_trail": {
                "total_entries": len(self.trail),
                "chain_verified": chain_valid,
                "chain_errors": chain_errors,
            },
        }

        # Log the assembly
        self.trail.log(
            agent=self.NAME,
            action="assemble_package",
            input_data=json.dumps({"controls": len(controls)}),
            output_data=json.dumps({"sections": list(package.keys())}),
            details=f"Assembled audit package: {len(evidence_index)} controls, "
                    f"{summary['total_policies']} policies, {summary['total_logs']} logs, "
                    f"{summary['total_notes']} notes",
        )

        return package


package_agent = PackageAgent(trail)
print("PackageAgent initialised")

## Orchestrator — Running the Full Workflow

The orchestrator coordinates all agents in sequence:
1. For each control → PolicyAgent, LogAgent, NoteAgent collect evidence
2. ChecklistAgent validates each control
3. PackageAgent assembles the final artifact

```
┌─────────────────────────────────────────────────────────────────┐
│                      ORCHESTRATOR                               │
│                                                                 │
│  For each control:                                              │
│  ┌──────────┐   ┌──────────┐   ┌──────────┐                    │
│  │ Policy   │──▶│  Log     │──▶│  Note    │──▶ evidence stored │
│  │ Agent    │   │  Agent   │   │  Agent   │                    │
│  └──────────┘   └──────────┘   └──────────┘                    │
│                                                                 │
│  Then:                                                          │
│  ┌──────────────┐   ┌──────────────┐                            │
│  │  Checklist   │──▶│   Package    │──▶ audit_package           │
│  │  Agent       │   │   Agent      │                            │
│  └──────────────┘   └──────────────┘                            │
│                                                                 │
│  Every action ──▶ AuditTrail (hash-chained)                     │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
def run_audit_workflow(controls, policy_agent, log_agent,
                       note_agent, checklist_agent, package_agent):
    """Execute the full audit evidence collection workflow."""

    print("=" * 65)
    print(f"  AUDIT WORKFLOW — {AUDIT_ID}")
    print(f"  Framework: {AUDIT_FRAMEWORK}")
    print(f"  Period: {AUDIT_PERIOD[0]} to {AUDIT_PERIOD[1]}")
    print("=" * 65)

    # Phase 1: Evidence Collection
    print("\n📋 PHASE 1: Evidence Collection")
    print("-" * 45)

    for cid, ctrl in controls.items():
        print(f"\n  Control {cid}: {ctrl.title}")

        # Collect policies
        policies = policy_agent.collect(ctrl)
        ctrl.policies = policies
        print(f"    Policies: {len(policies)} section(s) found")

        # Collect logs
        if "log" in ctrl.evidence_types:
            logs = log_agent.collect(ctrl)
            ctrl.logs = logs
            print(f"    Logs:     {len(logs)} entries found")

        # Collect notes
        notes = note_agent.collect(ctrl)
        ctrl.notes = notes
        print(f"    Notes:    {len(notes)} annotation(s) found")

    # Phase 2: Validation
    print("\n\n✅ PHASE 2: Checklist Validation")
    print("-" * 45)

    validation_results = []
    for cid, ctrl in controls.items():
        result = checklist_agent.validate(ctrl)
        validation_results.append(result)
        status_icon = "✓" if result["all_pass"] else "✗"
        finding_note = " (minor finding)" if result.get("has_minor_findings") else ""
        print(f"  {status_icon} {cid}: {result['status'].upper()}{finding_note}")

    # Phase 3: Package Assembly
    print("\n\n📦 PHASE 3: Audit Package Assembly")
    print("-" * 45)

    package = package_agent.assemble(controls, validation_results)
    print(f"  Controls:        {package['cover']['total_controls']}")
    print(f"  Total evidence:  {package['summary']['total_policies']} policies + "
          f"{package['summary']['total_logs']} logs + "
          f"{package['summary']['total_notes']} notes")
    print(f"  Evidenced:       {package['summary']['evidenced']}/{len(controls)}")
    print(f"  Minor findings:  {package['summary']['minor_findings']}")
    print(f"  Audit trail:     {package['audit_trail']['total_entries']} entries")
    print(f"  Chain verified:  {package['audit_trail']['chain_verified']}")
    print("\n" + "=" * 65)
    print("  WORKFLOW COMPLETE")
    print("=" * 65)

    return package


audit_package = run_audit_workflow(
    controls, policy_agent, log_agent,
    note_agent, checklist_agent, package_agent
)

## Audit Trail Inspection

Examining the full audit trail to demonstrate traceability. Every agent action is recorded with input/output hashes and chain linkage.

In [ ]:
trail_df = trail.to_dataframe()
print(f"Total audit trail entries: {len(trail_df)}")
print(f"\nAgent action counts:")
print(trail_df["agent"].value_counts().to_string())
print(f"\nStatus distribution:")
print(trail_df["status"].value_counts().to_string())

print(f"\n{'='*90}")
print(f"{'Seq':<5} {'Agent':<16} {'Action':<35} {'Status':<6} {'Chain Hash (first 16)'}")
print(f"{'='*90}")
for _, row in trail_df.iterrows():
    print(f"{row['seq']:<5} {row['agent']:<16} {row['action']:<35} "
          f"{row['status']:<6} {row['chain_hash'][:16]}...")

In [ ]:
# Verify chain integrity
is_valid, errors = trail.verify_chain()
print(f"Chain Integrity Verification")
print(f"{'='*40}")
print(f"Valid: {is_valid}")
if errors:
    for e in errors:
        print(f"  ERROR: {e}")
else:
    print("All chain hashes verified — no tampering detected.")

# Demonstrate tamper detection
print(f"\n--- Tamper Detection Demo ---")
trail_copy = copy.deepcopy(trail)
# Tamper with an entry
if len(trail_copy.entries) > 5:
    original_details = trail_copy.entries[5].details
    trail_copy.entries[5].details = "TAMPERED: I changed this entry"
    trail_copy.entries[5].output_hash = "0000000000000000000000000000000000000000000000000000000000000000"
    valid2, errors2 = trail_copy.verify_chain()
    print(f"After tampering entry #6:")
    print(f"  Valid: {valid2}")
    for e in errors2:
        print(f"  Detected: {e}")

## Evidence Index

A cross-referenced index showing what evidence was collected for each control — the core deliverable auditors review.

In [ ]:
evidence_df = pd.DataFrame(audit_package["evidence_index"])
print("EVIDENCE INDEX")
print("=" * 85)
print(evidence_df.to_string(index=False))

print(f"\n\nSUMMARY")
print("-" * 40)
for key, val in audit_package["summary"].items():
    print(f"  {key:<20s}: {val}")

## Detailed Control Evidence

Deep dive into a specific control to show the full evidence chain — from policy text to log entries to reviewer notes.

In [ ]:
def show_control_detail(ctrl: ComplianceControl):
    """Display full evidence detail for one control."""
    print(f"{'='*75}")
    print(f"  CONTROL: {ctrl.control_id} — {ctrl.title}")
    print(f"  Status:  {ctrl.status.value.upper()}")
    print(f"  TSC:     {', '.join(ctrl.tsc_criteria)}")
    print(f"{'='*75}")

    print(f"\n  DESCRIPTION:")
    print(f"    {ctrl.description}")

    print(f"\n  POLICY EVIDENCE ({len(ctrl.policies)} items):")
    for i, p in enumerate(ctrl.policies, 1):
        print(f"    [{i}] {p['doc_id']} — {p['doc_title']} (v{p['version']})")
        print(f"        Section {p['section']}: {p['text'][:100]}...")

    print(f"\n  LOG EVIDENCE ({len(ctrl.logs)} items):")
    for i, l in enumerate(ctrl.logs[:5], 1):  # show first 5
        print(f"    [{i}] {l['timestamp'][:19]} [{l['severity']}] {l['message'][:80]}")
    if len(ctrl.logs) > 5:
        print(f"    ... and {len(ctrl.logs) - 5} more log entries")

    print(f"\n  REVIEWER NOTES ({len(ctrl.notes)} items):")
    for i, n in enumerate(ctrl.notes, 1):
        print(f"    [{i}] {n['date']} — {n['author']}")
        print(f"        {n['text'][:100]}...")
        print(f"        Disposition: {n['disposition'].upper()}")

    print()


# Show detail for two controls
show_control_detail(controls["CC-6.1"])
show_control_detail(controls["CC-7.2"])

## Visual Analysis

Charts showing evidence distribution, agent activity, log severity patterns, and control status.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Evidence distribution by control
ev_df = pd.DataFrame(audit_package["evidence_index"])
x = range(len(ev_df))
width = 0.25
axes[0, 0].bar([i - width for i in x], ev_df["policies_count"], width,
               label="Policies", color="steelblue", alpha=0.8)
axes[0, 0].bar(x, ev_df["logs_count"], width,
               label="Logs", color="coral", alpha=0.8)
axes[0, 0].bar([i + width for i in x], ev_df["notes_count"], width,
               label="Notes", color="mediumseagreen", alpha=0.8)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(ev_df["control_id"], rotation=45, ha="right")
axes[0, 0].set_title("Evidence Count by Control")
axes[0, 0].legend()
axes[0, 0].set_ylabel("Count")

# 2. Control status pie chart
status_counts = ev_df["status"].value_counts()
colors_map = {"evidenced": "#2ecc71", "insufficient": "#e74c3c", "not_started": "#95a5a6"}
axes[0, 1].pie(status_counts.values,
               labels=[s.replace("_", " ").title() for s in status_counts.index],
               colors=[colors_map.get(s, "#3498db") for s in status_counts.index],
               autopct="%1.0f%%", startangle=90, textprops={"fontsize": 12})
axes[0, 1].set_title("Control Status Distribution")

# 3. Agent action counts
agent_counts = trail_df["agent"].value_counts()
axes[1, 0].barh(agent_counts.index, agent_counts.values, color="teal", alpha=0.8)
axes[1, 0].set_title("Audit Trail — Actions per Agent")
axes[1, 0].set_xlabel("Number of Actions")

# 4. Log severity over time
log_df = pd.DataFrame(LOG_STORE)
log_df["date"] = pd.to_datetime(log_df["timestamp"]).dt.date
sev_daily = log_df.groupby(["date", "severity"]).size().unstack(fill_value=0)
sev_daily.plot(ax=axes[1, 1], kind="area", alpha=0.6, stacked=True,
               color={"INFO": "steelblue", "WARN": "orange", "ERROR": "red"})
axes[1, 1].set_title("Log Volume by Severity Over Audit Period")
axes[1, 1].set_ylabel("Log Count")

plt.tight_layout()
plt.show()

In [ ]:
# Interactive evidence heatmap
ev_matrix = ev_df.set_index("control_id")[["policies_count", "logs_count", "notes_count"]]
ev_matrix.columns = ["Policies", "Logs", "Notes"]

fig = px.imshow(ev_matrix.values,
                x=ev_matrix.columns.tolist(),
                y=ev_matrix.index.tolist(),
                color_continuous_scale="YlGnBu",
                text_auto=True,
                title="Evidence Coverage Heatmap",
                labels=dict(x="Evidence Type", y="Control", color="Count"))
fig.update_layout(template="plotly_white", height=450)
fig.show()

## Audit Package Report

The final structured report that would be delivered to auditors.

In [ ]:
def print_audit_report(package: dict):
    """Print the formal audit package report."""
    cover = package["cover"]
    summary = package["summary"]
    trail_info = package["audit_trail"]

    print("╔" + "═" * 68 + "╗")
    print("║" + "AUDIT EVIDENCE PACKAGE".center(68) + "║")
    print("╠" + "═" * 68 + "╣")
    print(f"║  Audit ID:      {cover['audit_id']:<50s}║")
    print(f"║  Framework:     {cover['framework']:<50s}║")
    print(f"║  Organisation:  {cover['organisation']:<50s}║")
    print(f"║  Audit Period:  {cover['audit_period'][0]} to {cover['audit_period'][1]:<26s}║")
    print(f"║  Auditor:       {cover['auditor']:<50s}║")
    print(f"║  Prepared By:   {cover['prepared_by']:<50s}║")
    print(f"║  Prepared At:   {cover['prepared_at'][:19]:<50s}║")
    print("╠" + "═" * 68 + "╣")
    print("║" + "EVIDENCE SUMMARY".center(68) + "║")
    print("╠" + "═" * 68 + "╣")
    print(f"║  Total Controls:      {cover['total_controls']:<45d}║")
    print(f"║  Fully Evidenced:     {summary['evidenced']:<45d}║")
    print(f"║  Insufficient:        {summary['insufficient']:<45d}║")
    print(f"║  Policy Sections:     {summary['total_policies']:<45d}║")
    print(f"║  Log Entries:         {summary['total_logs']:<45d}║")
    print(f"║  Reviewer Notes:      {summary['total_notes']:<45d}║")
    print(f"║  Minor Findings:      {summary['minor_findings']:<45d}║")
    print("╠" + "═" * 68 + "╣")
    print("║" + "CONTROL STATUS".center(68) + "║")
    print("╠" + "═" * 68 + "╣")

    for entry in package["evidence_index"]:
        status_marker = "✓" if entry["status"] == "evidenced" else "✗"
        line = f"  {status_marker} {entry['control_id']}: {entry['title']}"
        ev_count = f"[{entry['total_evidence']} items]"
        padding = 68 - len(line) - len(ev_count) - 1
        print(f"║{line}{' ' * max(1, padding)}{ev_count}║")

    print("╠" + "═" * 68 + "╣")
    print("║" + "AUDIT TRAIL INTEGRITY".center(68) + "║")
    print("╠" + "═" * 68 + "╣")
    print(f"║  Trail Entries:       {trail_info['total_entries']:<45d}║")
    chain_str = "VERIFIED" if trail_info['chain_verified'] else "FAILED"
    print(f"║  Chain Integrity:     {chain_str:<45s}║")
    if trail_info["chain_errors"]:
        for err in trail_info["chain_errors"]:
            print(f"║    ERROR: {err:<57s}║")
    print("╚" + "═" * 68 + "╝")


print_audit_report(audit_package)

## Workflow Traceability Deep Dive

### How the Hash Chain Works

Each audit trail entry contains a `chain_hash` computed as:

$$\text{chain\_hash}_i = \text{SHA256}\big(\text{chain\_hash}_{i-1} \mathbin\| i \mathbin\| \text{agent} \mathbin\| \text{action} \mathbin\| \text{input\_hash} \mathbin\| \text{output\_hash}\big)$$

This creates a **Merkle-like chain** where:
- Modifying any entry invalidates all subsequent hashes
- The chain can be independently verified by any party with access to the trail
- Non-repudiation: the preparer cannot deny what actions were taken

### Why This Matters for Audit

| Property | How It Helps |
|---|---|
| **Completeness** | The trail shows every agent action — nothing was skipped |
| **Ordering** | Sequential IDs + timestamps prove the workflow order |
| **Integrity** | Hash chain detects any post-collection tampering |
| **Accountability** | Agent identity on each entry establishes responsibility |
| **Reproducibility** | Input/output hashes let auditors verify evidence hasn't changed |

In [ ]:
# Show first 5 entries with full hash details
print("DETAILED AUDIT TRAIL (first 5 entries)")
print("=" * 100)
for entry in trail.entries[:5]:
    print(f"\n  Seq:         {entry.seq}")
    print(f"  Timestamp:   {entry.timestamp}")
    print(f"  Agent:       {entry.agent}")
    print(f"  Action:      {entry.action}")
    print(f"  Status:      {entry.status}")
    print(f"  Details:     {entry.details}")
    print(f"  Input Hash:  {entry.input_hash}")
    print(f"  Output Hash: {entry.output_hash}")
    print(f"  Chain Hash:  {entry.chain_hash}")
    print(f"  {'─' * 60}")

## Log Analysis

Deeper analysis of the collected system logs — identifying patterns, warnings, and potential compliance concerns.

In [ ]:
log_df = pd.DataFrame(LOG_STORE)
log_df["timestamp"] = pd.to_datetime(log_df["timestamp"])
log_df["date"] = log_df["timestamp"].dt.date
log_df["hour"] = log_df["timestamp"].dt.hour

print("LOG ANALYSIS")
print("=" * 55)
print(f"Total entries:    {len(log_df)}")
print(f"Date range:       {log_df['date'].min()} to {log_df['date'].max()}")
print(f"Sources:          {log_df['source'].nunique()}")
print(f"\nBy Source:")
print(log_df["source"].value_counts().to_string())
print(f"\nBy Category:")
print(log_df["category"].value_counts().to_string())
print(f"\nBy Severity:")
print(log_df["severity"].value_counts().to_string())

# Warnings and errors
warnings_errors = log_df[log_df["severity"].isin(["WARN", "ERROR"])]
print(f"\n\nWARNINGS & ERRORS ({len(warnings_errors)} total):")
print("-" * 55)
for _, row in warnings_errors.head(10).iterrows():
    print(f"  [{row['severity']}] {str(row['date'])} | {row['source']}: {row['message'][:70]}")
if len(warnings_errors) > 10:
    print(f"  ... and {len(warnings_errors) - 10} more")

In [ ]:
# Log timeline by source
fig = px.histogram(log_df, x="timestamp", color="source",
                   nbins=30, title="Log Volume Timeline by Source",
                   labels={"timestamp": "Date", "count": "Entries"},
                   barmode="stack")
fig.update_layout(template="plotly_white", height=400)
fig.show()

## Findings Summary

Controls with minor findings or warnings that require attention in the audit response.

In [ ]:
print("FINDINGS SUMMARY")
print("=" * 70)

finding_num = 0
for cid, ctrl in controls.items():
    # Check for minor findings in notes
    for note in ctrl.notes:
        if note.get("disposition") == "minor_finding":
            finding_num += 1
            print(f"\n  Finding #{finding_num}")
            print(f"  Control:     {cid} — {ctrl.title}")
            print(f"  Author:      {note['author']}")
            print(f"  Date:        {note['date']}")
            print(f"  Description: {note['text'][:120]}...")
            print(f"  Disposition: MINOR FINDING")

    # Check for warning logs
    warn_logs = [l for l in ctrl.logs if l.get("severity") in ("WARN", "ERROR")]
    if warn_logs:
        finding_num += 1
        print(f"\n  Finding #{finding_num}")
        print(f"  Control:     {cid} — {ctrl.title}")
        print(f"  Source:      System logs")
        print(f"  Description: {len(warn_logs)} warning/error log entries detected")
        for wl in warn_logs[:3]:
            print(f"    - [{wl['severity']}] {wl['message'][:70]}")
        if len(warn_logs) > 3:
            print(f"    ... and {len(warn_logs) - 3} more")

if finding_num == 0:
    print("  No findings detected.")
else:
    print(f"\n  Total findings: {finding_num}")

## Evaluation & Quality Checks

Programmatic verification that the audit package meets quality standards.

In [ ]:
print("AUDIT PACKAGE QUALITY CHECKS")
print("=" * 55)

checks_passed = 0
checks_total = 0

# 1. All controls have at least one evidence item
checks_total += 1
all_have_evidence = all(
    (len(c.policies) + len(c.logs) + len(c.notes)) > 0
    for c in controls.values()
)
status = "PASS" if all_have_evidence else "FAIL"
print(f"  [{status}] All controls have at least one evidence item")
checks_passed += int(all_have_evidence)

# 2. Audit trail chain is valid
checks_total += 1
chain_ok, _ = trail.verify_chain()
status = "PASS" if chain_ok else "FAIL"
print(f"  [{status}] Audit trail chain integrity verified")
checks_passed += int(chain_ok)

# 3. All controls have been validated
checks_total += 1
all_validated = all(
    c.status != ControlStatus.NOT_STARTED
    for c in controls.values()
)
status = "PASS" if all_validated else "FAIL"
print(f"  [{status}] All controls have been validated by ChecklistAgent")
checks_passed += int(all_validated)

# 4. No controls are insufficient
checks_total += 1
none_insufficient = not any(
    c.status == ControlStatus.INSUFFICIENT
    for c in controls.values()
)
status = "PASS" if none_insufficient else "WARN"
print(f"  [{status}] No controls have insufficient evidence")
checks_passed += int(none_insufficient)

# 5. Audit trail has entries from all agents
checks_total += 1
expected_agents = {"PolicyAgent", "LogAgent", "NoteAgent", "ChecklistAgent", "PackageAgent"}
actual_agents = set(trail_df["agent"].unique())
all_agents = expected_agents.issubset(actual_agents)
status = "PASS" if all_agents else "FAIL"
print(f"  [{status}] All 5 agents recorded in audit trail")
checks_passed += int(all_agents)

# 6. Every control has policy evidence
checks_total += 1
all_have_policies = all(len(c.policies) >= 1 for c in controls.values())
status = "PASS" if all_have_policies else "FAIL"
print(f"  [{status}] Every control has at least one policy reference")
checks_passed += int(all_have_policies)

# 7. Package has all required sections
checks_total += 1
required_sections = {"cover", "evidence_index", "validation_results", "summary", "audit_trail"}
has_sections = required_sections.issubset(set(audit_package.keys()))
status = "PASS" if has_sections else "FAIL"
print(f"  [{status}] Package contains all required sections")
checks_passed += int(has_sections)

# 8. Timestamps are in ISO 8601 format
checks_total += 1
import re as _re
iso_pattern = _re.compile(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}")
all_iso = all(
    iso_pattern.match(e.timestamp)
    for e in trail.entries
)
status = "PASS" if all_iso else "FAIL"
print(f"  [{status}] All audit trail timestamps are ISO 8601")
checks_passed += int(all_iso)

print(f"\n  Result: {checks_passed}/{checks_total} checks passed")

## How to Extend This Agent

### Production Upgrades

1. **Connect real data sources**: Replace simulated stores with API calls to ServiceNow (tickets), Splunk/Elastic (logs), Confluence (policies), Jira (notes)
2. **Digital signatures**: Add GPG/PGP signing to the audit package so the preparer's identity is cryptographically verified
3. **Evidence attachment**: Store actual file artifacts (PDFs, screenshots) alongside metadata references
4. **Continuous compliance**: Run the workflow on a schedule (monthly, quarterly) and diff against previous packages
5. **Multi-framework**: Extend `ComplianceControl` to map to multiple frameworks simultaneously (SOC 2 + ISO 27001 + PCI-DSS)

### Architecture Patterns

| Pattern | When to Use |
|---|---|
| **Sequential chain** (this notebook) | When evidence types depend on each other — notes reference logs |
| **Parallel collection** | When agents are independent — collect policies, logs, notes simultaneously |
| **Hierarchical** | When controls nest — a parent control delegates to sub-control agents |
| **Reviewer-in-the-loop** | When human sign-off is required between evidence collection and package assembly |

## Common Mistakes

1. **Collecting evidence without timestamps**: Auditors cannot verify timeliness without ISO-8601 timestamps on every artifact
2. **No chain of custody**: Gathering evidence into a folder without recording who collected it, when, and from what source
3. **Ignoring version control on policies**: Citing a policy without its version number means the auditor cannot verify which requirements applied during the audit period
4. **Over-relying on log volume**: 1000 log entries do not prove a control works — auditors look for specific event types at specific times
5. **Skipping the validation step**: Assembling evidence without verifying sufficiency against each control's requirements
6. **Mutable audit trails**: Using a regular list instead of a hash chain — entries can be silently deleted or modified after the fact

## Mini Challenge / Exercises

1. **Add a new control**: Define `CC-6.6` (encryption at rest) with its own policy, log templates, and reviewer note. Run the workflow and verify the package includes it
2. **Implement severity filtering**: Modify `LogAgent.collect()` to accept a `max_severity` parameter and test collecting only WARNs and ERRORs
3. **Build a gap report**: Write a function that identifies controls where *no* reviewer note exists and generates an actionable gap report
4. **Export to JSON**: Serialise the full `audit_package` to a JSON file on disk with proper indentation — this is the artifact an auditor would receive
5. **Add digital signing**: Compute a SHA-256 hash of the entire serialised package and store it as a "package_hash" field — this is the first step toward non-repudiation

## Final Summary & Key Takeaways

### What We Built
- A **5-agent audit compliance system**: PolicyAgent, LogAgent, NoteAgent, ChecklistAgent, PackageAgent
- **Hash-chained audit trail** with tamper detection using SHA-256
- **8 SOC 2 compliance controls** with realistic evidence from simulated enterprise systems
- **200 structured log entries** across 7 source systems
- **Evidence sufficiency validation** with pass/fail per control
- **Formal audit package** with cover page, evidence index, and integrity verification

### Key Takeaways
1. **Traceability is non-negotiable**: Every agent action must be recorded with timestamps, identity, and content hashes
2. **Hash chains provide tamper evidence**: A single modified entry invalidates all subsequent hashes — auditors can verify independently
3. **Evidence has three pillars**: Policy (what should happen), Logs (what did happen), Notes (human judgement on the gap)
4. **Validation before packaging**: The ChecklistAgent ensures no control ships with insufficient evidence
5. **Agent separation enables audit**: Each agent has a single responsibility — if a finding is tied to log evidence, the LogAgent's trail entries show exactly how that evidence was collected

---
*Educational notebook — part of the NLP / Agentic Workflows portfolio.*